# 04 — Athlete-Disjoint Landmark Modelling (SoccerMon 2020)

This notebook performs the canonical statistical and machine-learning analyses for the fixed-landmark athlete-session formulation.

## Target and interpretation

The outcome is a **session-level injury-associated indicator**. Exact within-session injury-onset timestamps are unavailable. Therefore, the analyses below do **not** estimate minute-specific injury risk, localize injury onset, or evaluate an operational alerting policy.

At each landmark (10, 20, 30, 40, 50, and 60 minutes), exactly one representation is selected per athlete-session using features constructed only from information observed by that session-clock point. The question is whether discrimination of the same athlete-session target changes as more within-session information becomes available.

## Evaluation design

- Primary modelling cohort: Team A only.
- Five deterministic athlete-disjoint outer folds.
- Exactly one athlete with at least one positive session is held out in each fold.
- All preprocessing is fitted on outer-training athletes only.
- Five held-out folds are pooled so each evaluable athlete-session contributes one out-of-athlete prediction per landmark.
- Uncertainty uses athlete-cluster bootstrap resampling of pooled out-of-fold predictions.

## Provenance note

An older archived execution of the predecessor notebook produced Random Forest and XGBoost point estimates that do not exactly reproduce under the current software environment, while Logistic Regression reproduces exactly. Those historical tree-model outputs are **not** treated as canonical here. This notebook recomputes all model-family benchmarks in one environment, records package versions, and exports the resulting out-of-fold predictions.

Two robustness analyses that were described in an earlier manuscript but were not present as executable cells in the archived notebook are reconstructed explicitly here:

1. a common-cohort sensitivity restricted to athlete-sessions observable through 60 minutes; and
2. 100 alternative allocations of negative athletes across the five outer folds while keeping one positive athlete per fold.

These reconstructed analyses are part of the current reproducible pipeline and should be used for any future manuscript revision.


In [ ]:
from pathlib import Path
import platform

import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    import xgboost
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    xgboost = None
    XGBClassifier = None
    XGBOOST_AVAILABLE = False

RANDOM_STATE = 42
LANDMARKS = [10, 20, 30, 40, 50, 60]

print("Python:", platform.python_version())
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print(
    "xgboost:",
    xgboost.__version__ if XGBOOST_AVAILABLE else "not installed",
)


## 1. Repository paths and canonical inputs

Notebook 03 provides the modelling table and the exact ordered 63-feature primary representation.


In [ ]:
def find_project_root(start: Path) -> Path:
    """Resolve the repository root when run from the root or notebooks directory."""
    start = start.resolve()
    if start.name.lower() == "notebooks":
        return start.parent
    return start


PROJECT_ROOT = find_project_root(Path.cwd())

INPUT_DIR = PROJECT_ROOT / "results" / "modelling_data"
OUTPUT_DIR = PROJECT_ROOT / "results" / "baseline"
FIGURE_DIR = OUTPUT_DIR / "figures"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FILE = INPUT_DIR / "model_df_2020.csv"
FEATURE_FILE = INPUT_DIR / "primary_cum_dyn_features_2020.csv"

for required_file in [MODEL_FILE, FEATURE_FILE]:
    if not required_file.exists():
        raise FileNotFoundError(
            "Required modelling input was not found. "
            "Run Notebook 03 successfully before Notebook 04. "
            f"Missing: {required_file}"
        )

print("Project root:", PROJECT_ROOT)
print("Model input:", MODEL_FILE)
print("Feature input:", FEATURE_FILE)
print("Output directory:", OUTPUT_DIR)


## 2. Load modelling resource and freeze feature families

In [ ]:
df = pd.read_csv(MODEL_FILE, low_memory=False)

feature_table = (
    pd.read_csv(FEATURE_FILE)
    .sort_values("feature_order")
    .reset_index(drop=True)
)

primary_features = feature_table["feature_name"].tolist()

pre_session_features = [
    "ctl28",
    "ctl42",
    "daily_load",
    "weekly_load",
    "acwr",
    "monotony",
    "strain",
    "sleep_duration",
    "sleep_quality",
    "fatigue",
    "mood",
    "readiness",
    "soreness",
    "stress",
]

cumulative_features = [
    feature for feature in primary_features
    if "_cum_" in feature
]

dynamic_features = [
    feature for feature in primary_features
    if feature not in cumulative_features
]

assert len(pre_session_features) == 14
assert len(cumulative_features) == 30
assert len(dynamic_features) == 33
assert len(primary_features) == 63
assert len(set(primary_features)) == 63

missing = (
    set(pre_session_features)
    | set(primary_features)
    | {"player_name", "session_id", "minute_idx", "injury"}
) - set(df.columns)

assert not missing, f"Missing modelling columns: {sorted(missing)}"

df["team"] = (
    df["player_name"]
    .astype(str)
    .str.split("-", n=1)
    .str[0]
)

print("Modelling rows:", len(df))
print("Primary features:", len(primary_features))
print("PRE / CUM / DYN:", 14, 30, 33)


## 3. Primary Team-A cohort and deterministic outer folds

All positive athlete-sessions occur in Team A. Team B therefore cannot serve as an injury-positive external discrimination cohort and is excluded from the supervised model benchmark.

The fold construction is deterministic: positive athletes are sorted and assigned one per fold; negative athletes are sorted and split into approximately equal-sized chunks.


In [ ]:
session_table = (
    df[["player_name", "session_id", "injury", "team"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

team_a_sessions = (
    session_table.loc[
        session_table["team"] == "TeamA"
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(team_a_sessions) == 2_259
assert team_a_sessions["player_name"].nunique() == 27
assert int(team_a_sessions["injury"].sum()) == 22

athlete_outcome = (
    team_a_sessions
    .groupby("player_name", as_index=False)
    .agg(n_positive=("injury", "sum"))
)

positive_athletes = sorted(
    athlete_outcome.loc[
        athlete_outcome["n_positive"] > 0,
        "player_name",
    ].tolist()
)

negative_athletes = sorted(
    athlete_outcome.loc[
        athlete_outcome["n_positive"] == 0,
        "player_name",
    ].tolist()
)

assert len(positive_athletes) == 5
assert len(negative_athletes) == 22

negative_chunks = np.array_split(
    negative_athletes,
    len(positive_athletes),
)

outer_folds = []

for fold_idx in range(5):
    test_athletes = (
        [positive_athletes[fold_idx]]
        + list(negative_chunks[fold_idx])
    )

    train_athletes = [
        athlete
        for athlete in athlete_outcome["player_name"].tolist()
        if athlete not in test_athletes
    ]

    outer_folds.append(
        {
            "fold": fold_idx + 1,
            "train_athletes": train_athletes,
            "test_athletes": test_athletes,
        }
    )

fold_rows = []

for fold in outer_folds:
    for athlete in fold["test_athletes"]:
        fold_rows.append(
            {
                "player_name": athlete,
                "test_fold": fold["fold"],
                "positive_athlete": int(
                    athlete in positive_athletes
                ),
            }
        )

fold_assignment = pd.DataFrame(fold_rows)

assert len(fold_assignment) == 27
assert fold_assignment["player_name"].nunique() == 27
assert (
    fold_assignment
    .groupby("test_fold")["positive_athlete"]
    .sum()
    .eq(1)
    .all()
)

display(
    fold_assignment
    .sort_values(["test_fold", "positive_athlete", "player_name"])
    .reset_index(drop=True)
)


## 4. Landmark datasets

Each landmark contains at most one row per athlete-session. No interpolation is performed: an athlete-session contributes only when that exact elapsed-time row exists.


In [ ]:
team_a_athletes = set(team_a_sessions["player_name"])

all_ablation_features = list(
    dict.fromkeys(
        pre_session_features
        + cumulative_features
        + dynamic_features
    )
)

landmark_datasets = {}
landmark_rows = []

for landmark in LANDMARKS:
    landmark_df = (
        df.loc[
            (df["minute_idx"] == landmark)
            & (df["player_name"].isin(team_a_athletes)),
            ["player_name", "session_id", "injury"]
            + all_ablation_features,
        ]
        .copy()
        .reset_index(drop=True)
    )

    assert not landmark_df.duplicated(
        ["player_name", "session_id"]
    ).any()

    assert int(landmark_df["injury"].sum()) == 22
    assert (
        landmark_df.loc[
            landmark_df["injury"] == 1,
            "player_name",
        ].nunique()
        == 5
    )

    landmark_datasets[landmark] = landmark_df

    landmark_rows.append(
        {
            "landmark": landmark,
            "athlete_sessions": len(landmark_df),
            "athletes": landmark_df["player_name"].nunique(),
            "positive_sessions": int(
                landmark_df["injury"].sum()
            ),
            "prevalence": float(
                landmark_df["injury"].mean()
            ),
        }
    )

landmark_availability = pd.DataFrame(landmark_rows)

display(landmark_availability)


## 5. Model factories and reusable evaluation functions

The primary classifier is L2-regularized Logistic Regression. Median imputation and standardization are fitted only on outer-training observations.

Random Forest and XGBoost are fixed benchmark learners. No data-dependent hyperparameter search is performed.


In [ ]:
def make_logistic_model():
    return Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (
                "classifier",
                LogisticRegression(
                    C=1.0,
                    class_weight="balanced",
                    solver="liblinear",
                    max_iter=2000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


def make_random_forest_model():
    return Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            (
                "classifier",
                RandomForestClassifier(
                    n_estimators=500,
                    max_depth=None,
                    min_samples_leaf=2,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    )


def make_xgboost_model(y_train):
    if not XGBOOST_AVAILABLE:
        raise RuntimeError(
            "XGBoost is not installed in the current environment."
        )

    n_positive = int((y_train == 1).sum())
    n_negative = int((y_train == 0).sum())

    if n_positive == 0:
        raise ValueError("XGBoost training fold contains no positive sessions.")

    return Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            (
                "classifier",
                XGBClassifier(
                    n_estimators=300,
                    max_depth=3,
                    learning_rate=0.03,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    eval_metric="logloss",
                    scale_pos_weight=n_negative / n_positive,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    )


def evaluate_oof(
    landmark_df,
    features,
    folds,
    model_name="logistic",
):
    """Return pooled athlete-disjoint OOF predictions for one landmark."""
    fold_predictions = []

    for fold in folds:
        train = landmark_df.loc[
            landmark_df["player_name"].isin(
                fold["train_athletes"]
            )
        ].copy()

        test = landmark_df.loc[
            landmark_df["player_name"].isin(
                fold["test_athletes"]
            )
        ].copy()

        assert set(train["player_name"]).isdisjoint(
            set(test["player_name"])
        )

        X_train = train[features]
        y_train = train["injury"].astype(int)
        X_test = test[features]
        y_test = test["injury"].astype(int)

        assert y_train.nunique() == 2
        assert y_test.nunique() == 2

        if model_name == "logistic":
            model = make_logistic_model()
        elif model_name == "random_forest":
            model = make_random_forest_model()
        elif model_name == "xgboost":
            model = make_xgboost_model(y_train)
        else:
            raise ValueError(f"Unknown model_name: {model_name}")

        model.fit(X_train, y_train)

        probabilities = model.predict_proba(X_test)[:, 1]

        fold_predictions.append(
            pd.DataFrame(
                {
                    "player_name": test["player_name"].to_numpy(),
                    "session_id": test["session_id"].to_numpy(),
                    "injury": y_test.to_numpy(),
                    "y_prob": probabilities,
                    "fold": fold["fold"],
                }
            )
        )

    oof = pd.concat(
        fold_predictions,
        ignore_index=True,
    )

    assert not oof.duplicated(
        ["player_name", "session_id"]
    ).any()

    assert len(oof) == len(landmark_df)

    return oof


def summarize_oof(oof):
    prevalence = float(oof["injury"].mean())

    return {
        "roc_auc": roc_auc_score(
            oof["injury"],
            oof["y_prob"],
        ),
        "pr_auc": average_precision_score(
            oof["injury"],
            oof["y_prob"],
        ),
        "prevalence": prevalence,
        "pr_over_baseline": (
            average_precision_score(
                oof["injury"],
                oof["y_prob"],
            )
            / prevalence
        ),
        "brier": brier_score_loss(
            oof["injury"],
            oof["y_prob"],
        ),
    }


## 6. Primary CUM+DYN Logistic Regression analysis

The six landmark models are fitted independently using identical outer folds and the fixed 63-feature representation.


In [ ]:
primary_oof = {}
primary_rows = []

for landmark in LANDMARKS:
    oof = evaluate_oof(
        landmark_datasets[landmark],
        primary_features,
        outer_folds,
        model_name="logistic",
    )

    metrics = summarize_oof(oof)

    primary_oof[landmark] = oof

    primary_rows.append(
        {
            "landmark": landmark,
            **metrics,
        }
    )

primary_results = pd.DataFrame(primary_rows)

EXPECTED_PRIMARY_ROC = np.array(
    [0.499349, 0.556940, 0.607200, 0.427717, 0.401846, 0.367348]
)

EXPECTED_PRIMARY_PR = np.array(
    [0.010696, 0.014999, 0.012902, 0.008785, 0.007913, 0.008001]
)

np.testing.assert_allclose(
    primary_results["roc_auc"].to_numpy(),
    EXPECTED_PRIMARY_ROC,
    atol=5e-6,
    rtol=0.0,
)

np.testing.assert_allclose(
    primary_results["pr_auc"].to_numpy(),
    EXPECTED_PRIMARY_PR,
    atol=5e-6,
    rtol=0.0,
)

display(primary_results.round(6))

print("Primary Logistic Regression results reproduce the frozen baseline.")


## 7. Athlete-cluster bootstrap uncertainty

The bootstrap resamples athletes from the pooled OOF predictions. It quantifies cluster-level sampling uncertainty conditional on the fitted cross-validation procedure; it does not refit the full modelling pipeline inside each replicate.


In [ ]:
def cluster_bootstrap_metrics(
    oof,
    n_boot=1000,
    seed=2026,
):
    rng = np.random.default_rng(seed)
    athletes = np.array(
        sorted(oof["player_name"].unique())
    )

    roc_values = []
    pr_values = []

    for _ in range(n_boot):
        sampled = rng.choice(
            athletes,
            size=len(athletes),
            replace=True,
        )

        parts = []

        for draw_index, athlete in enumerate(sampled):
            part = oof.loc[
                oof["player_name"] == athlete
            ].copy()

            part["bootstrap_cluster"] = draw_index
            parts.append(part)

        boot = pd.concat(
            parts,
            ignore_index=True,
        )

        if boot["injury"].nunique() < 2:
            continue

        roc_values.append(
            roc_auc_score(
                boot["injury"],
                boot["y_prob"],
            )
        )
        pr_values.append(
            average_precision_score(
                boot["injury"],
                boot["y_prob"],
            )
        )

    return {
        "n_valid": len(roc_values),
        "roc_ci_low": np.quantile(roc_values, 0.025),
        "roc_ci_high": np.quantile(roc_values, 0.975),
        "pr_ci_low": np.quantile(pr_values, 0.025),
        "pr_ci_high": np.quantile(pr_values, 0.975),
    }


primary_uncertainty_rows = []

for landmark in LANDMARKS:
    uncertainty = cluster_bootstrap_metrics(
        primary_oof[landmark],
        n_boot=1000,
        seed=2026 + landmark,
    )

    primary_uncertainty_rows.append(
        {
            "landmark": landmark,
            **uncertainty,
        }
    )

primary_uncertainty = (
    primary_results
    .merge(
        pd.DataFrame(primary_uncertainty_rows),
        on="landmark",
        validate="one_to_one",
    )
)

display(primary_uncertainty.round(6))


## 8. Common-cohort landmark sensitivity

To separate representation changes from landmark-specific attrition, all six landmarks are re-evaluated on the fixed athlete-session subset that remains observable at 60 minutes.


In [ ]:
common_keys = (
    landmark_datasets[60][
        ["player_name", "session_id"]
    ]
    .drop_duplicates()
)

assert len(common_keys) == 2_104

common_cohort_oof = {}
common_cohort_rows = []

for landmark in LANDMARKS:
    dx = (
        landmark_datasets[landmark]
        .merge(
            common_keys,
            on=["player_name", "session_id"],
            how="inner",
            validate="one_to_one",
        )
        .reset_index(drop=True)
    )

    assert len(dx) == 2_104
    assert int(dx["injury"].sum()) == 22

    oof = evaluate_oof(
        dx,
        primary_features,
        outer_folds,
        model_name="logistic",
    )

    metrics = summarize_oof(oof)

    common_cohort_oof[landmark] = oof
    common_cohort_rows.append(
        {
            "landmark": landmark,
            **metrics,
        }
    )

common_cohort_results = pd.DataFrame(
    common_cohort_rows
)

display(common_cohort_results.round(6))


## 9. Alternative negative-athlete fold-allocation sensitivity

The five positive athletes remain deterministically assigned one per test fold. For each of 100 prespecified random allocations, the 22 negative athletes are permuted and split into five approximately equal chunks.

This analysis tests whether the primary landmark pattern is an artefact of one deterministic allocation of negative athletes.


In [ ]:
def make_alternative_folds(
    negative_athletes,
    positive_athletes,
    seed,
):
    rng = np.random.default_rng(seed)

    shuffled_negatives = list(
        rng.permutation(negative_athletes)
    )

    chunks = np.array_split(
        shuffled_negatives,
        len(positive_athletes),
    )

    all_athletes = (
        positive_athletes
        + negative_athletes
    )

    folds = []

    for fold_idx in range(5):
        test_athletes = (
            [positive_athletes[fold_idx]]
            + list(chunks[fold_idx])
        )

        train_athletes = [
            athlete
            for athlete in all_athletes
            if athlete not in test_athletes
        ]

        folds.append(
            {
                "fold": fold_idx + 1,
                "train_athletes": train_athletes,
                "test_athletes": test_athletes,
            }
        )

    return folds


N_ALTERNATIVE_ALLOCATIONS = 100
ALLOCATION_SEED_BASE = 202600

allocation_rows = []

for allocation_index in range(
    N_ALTERNATIVE_ALLOCATIONS
):
    folds = make_alternative_folds(
        negative_athletes,
        positive_athletes,
        seed=ALLOCATION_SEED_BASE + allocation_index,
    )

    for landmark in LANDMARKS:
        oof = evaluate_oof(
            landmark_datasets[landmark],
            primary_features,
            folds,
            model_name="logistic",
        )

        metrics = summarize_oof(oof)

        allocation_rows.append(
            {
                "allocation": allocation_index + 1,
                "landmark": landmark,
                "roc_auc": metrics["roc_auc"],
                "pr_auc": metrics["pr_auc"],
            }
        )

alternative_fold_results = pd.DataFrame(
    allocation_rows
)

alternative_fold_summary = (
    alternative_fold_results
    .groupby("landmark", as_index=False)
    .agg(
        roc_median=("roc_auc", "median"),
        roc_q025=(
            "roc_auc",
            lambda x: np.quantile(x, 0.025),
        ),
        roc_q975=(
            "roc_auc",
            lambda x: np.quantile(x, 0.975),
        ),
        pr_median=("pr_auc", "median"),
        pr_q025=(
            "pr_auc",
            lambda x: np.quantile(x, 0.025),
        ),
        pr_q975=(
            "pr_auc",
            lambda x: np.quantile(x, 0.975),
        ),
    )
)

display(alternative_fold_summary.round(6))


## 10. Equal-athlete-weight sensitivity

The primary estimand pools athlete-sessions. This sensitivity instead gives every athlete equal total weight within a landmark by assigning each of that athlete's sessions weight `1 / n_sessions_for_that_athlete`.


In [ ]:
equal_athlete_rows = []

for landmark in LANDMARKS:
    oof = primary_oof[landmark].copy()

    session_counts = (
        oof.groupby("player_name")["session_id"]
        .transform("size")
    )

    weights = 1.0 / session_counts

    equal_athlete_rows.append(
        {
            "landmark": landmark,
            "roc_auc_equal_athlete": roc_auc_score(
                oof["injury"],
                oof["y_prob"],
                sample_weight=weights,
            ),
            "pr_auc_equal_athlete": average_precision_score(
                oof["injury"],
                oof["y_prob"],
                sample_weight=weights,
            ),
            "weighted_prevalence": np.average(
                oof["injury"],
                weights=weights,
            ),
        }
    )

equal_athlete_results = pd.DataFrame(
    equal_athlete_rows
)

display(equal_athlete_results.round(6))


## 11. Controlled feature-family ablation

Cohort, landmarks, outer folds, preprocessing, classifier, and class weighting are fixed. Only the feature representation changes.


In [ ]:
ablation_feature_sets = {
    "PRE": pre_session_features,
    "CUM": cumulative_features,
    "DYN": dynamic_features,
    "PRE+CUM": pre_session_features + cumulative_features,
    "PRE+DYN": pre_session_features + dynamic_features,
    "CUM+DYN": primary_features,
    "ALL": all_ablation_features,
}

expected_counts = {
    "PRE": 14,
    "CUM": 30,
    "DYN": 33,
    "PRE+CUM": 44,
    "PRE+DYN": 47,
    "CUM+DYN": 63,
    "ALL": 77,
}

for name, features in ablation_feature_sets.items():
    assert len(features) == expected_counts[name]
    assert len(set(features)) == expected_counts[name]

ablation_oof = {}
ablation_rows = []

for config_name, features in ablation_feature_sets.items():
    ablation_oof[config_name] = {}

    for landmark in LANDMARKS:
        oof = evaluate_oof(
            landmark_datasets[landmark],
            features,
            outer_folds,
            model_name="logistic",
        )

        metrics = summarize_oof(oof)

        ablation_oof[config_name][landmark] = oof

        ablation_rows.append(
            {
                "configuration": config_name,
                "landmark": landmark,
                "n_features": len(features),
                **metrics,
            }
        )

ablation_results = pd.DataFrame(
    ablation_rows
)

# CUM+DYN must exactly reproduce the primary model.
for landmark in LANDMARKS:
    original = (
        primary_oof[landmark]
        .sort_values(["player_name", "session_id"])
        .reset_index(drop=True)
    )

    replicated = (
        ablation_oof["CUM+DYN"][landmark]
        .sort_values(["player_name", "session_id"])
        .reset_index(drop=True)
    )

    np.testing.assert_allclose(
        original["y_prob"].to_numpy(),
        replicated["y_prob"].to_numpy(),
        atol=1e-12,
        rtol=0.0,
    )

display(
    ablation_results
    .pivot(
        index="configuration",
        columns="landmark",
        values="roc_auc",
    )
    .round(3)
)

display(
    ablation_results
    .pivot(
        index="configuration",
        columns="landmark",
        values="pr_auc",
    )
    .round(4)
)


## 12. Model-family benchmark in the current environment

Logistic Regression, Random Forest, and—when installed—XGBoost use the exact same landmark datasets, 63 CUM+DYN features, and athlete-disjoint outer folds.

Historical RF/XGBoost point estimates from an earlier archived execution are not imported into this analysis. The table below is recomputed from scratch in the active environment.


In [ ]:
benchmark_models = [
    "logistic",
    "random_forest",
]

if XGBOOST_AVAILABLE:
    benchmark_models.append("xgboost")

benchmark_oof = {}
benchmark_rows = []

for model_name in benchmark_models:
    benchmark_oof[model_name] = {}

    for landmark in LANDMARKS:
        oof = evaluate_oof(
            landmark_datasets[landmark],
            primary_features,
            outer_folds,
            model_name=model_name,
        )

        metrics = summarize_oof(oof)

        benchmark_oof[model_name][landmark] = oof

        benchmark_rows.append(
            {
                "model": model_name,
                "landmark": landmark,
                **metrics,
            }
        )

benchmark_results = pd.DataFrame(
    benchmark_rows
)

display(
    benchmark_results
    .pivot(
        index="model",
        columns="landmark",
        values="roc_auc",
    )
    .round(6)
)

display(
    benchmark_results
    .pivot(
        index="model",
        columns="landmark",
        values="pr_auc",
    )
    .round(6)
)


## 13. Paired athlete-cluster bootstrap comparisons

Comparisons use matched athlete-session predictions and shared cluster-resampling draws within each contrast. Intervals are descriptive sensitivity contrasts; no multiplicity-adjusted confirmatory testing procedure was prespecified.


In [ ]:
def paired_cluster_bootstrap(
    reference,
    comparator,
    n_boot=1000,
    seed=2026,
):
    ref = (
        reference[
            ["player_name", "session_id", "injury", "y_prob"]
        ]
        .rename(columns={"y_prob": "p_reference"})
    )

    comp = (
        comparator[
            ["player_name", "session_id", "injury", "y_prob"]
        ]
        .rename(columns={"y_prob": "p_comparator"})
    )

    paired = ref.merge(
        comp,
        on=["player_name", "session_id", "injury"],
        how="inner",
        validate="one_to_one",
    )

    assert len(paired) == len(ref) == len(comp)

    observed_delta_roc = (
        roc_auc_score(
            paired["injury"],
            paired["p_comparator"],
        )
        - roc_auc_score(
            paired["injury"],
            paired["p_reference"],
        )
    )

    observed_delta_pr = (
        average_precision_score(
            paired["injury"],
            paired["p_comparator"],
        )
        - average_precision_score(
            paired["injury"],
            paired["p_reference"],
        )
    )

    athletes = np.array(
        sorted(paired["player_name"].unique())
    )
    rng = np.random.default_rng(seed)

    delta_roc = []
    delta_pr = []

    for _ in range(n_boot):
        sampled = rng.choice(
            athletes,
            size=len(athletes),
            replace=True,
        )

        parts = []

        for draw_index, athlete in enumerate(sampled):
            part = paired.loc[
                paired["player_name"] == athlete
            ].copy()

            part["bootstrap_cluster"] = draw_index
            parts.append(part)

        boot = pd.concat(parts, ignore_index=True)

        if boot["injury"].nunique() < 2:
            continue

        delta_roc.append(
            roc_auc_score(
                boot["injury"],
                boot["p_comparator"],
            )
            - roc_auc_score(
                boot["injury"],
                boot["p_reference"],
            )
        )

        delta_pr.append(
            average_precision_score(
                boot["injury"],
                boot["p_comparator"],
            )
            - average_precision_score(
                boot["injury"],
                boot["p_reference"],
            )
        )

    return {
        "delta_roc": observed_delta_roc,
        "delta_roc_ci_low": np.quantile(delta_roc, 0.025),
        "delta_roc_ci_high": np.quantile(delta_roc, 0.975),
        "delta_pr": observed_delta_pr,
        "delta_pr_ci_low": np.quantile(delta_pr, 0.025),
        "delta_pr_ci_high": np.quantile(delta_pr, 0.975),
        "n_valid_boot": len(delta_roc),
    }


ablation_comparison_rows = []

for config_name in [
    "PRE",
    "CUM",
    "DYN",
    "PRE+CUM",
    "PRE+DYN",
    "ALL",
]:
    for landmark in LANDMARKS:
        result = paired_cluster_bootstrap(
            ablation_oof["CUM+DYN"][landmark],
            ablation_oof[config_name][landmark],
            n_boot=1000,
            seed=10_000 + landmark,
        )

        ablation_comparison_rows.append(
            {
                "comparator": config_name,
                "landmark": landmark,
                **result,
            }
        )

paired_ablation_results = pd.DataFrame(
    ablation_comparison_rows
)

model_comparison_rows = []

for comparator_name in [
    model
    for model in benchmark_models
    if model != "logistic"
]:
    for landmark in LANDMARKS:
        result = paired_cluster_bootstrap(
            benchmark_oof["logistic"][landmark],
            benchmark_oof[comparator_name][landmark],
            n_boot=1000,
            seed=20_000 + landmark,
        )

        model_comparison_rows.append(
            {
                "comparator": comparator_name,
                "landmark": landmark,
                **result,
            }
        )

paired_model_results = pd.DataFrame(
    model_comparison_rows
)

display(paired_ablation_results.round(6))
display(paired_model_results.round(6))


## 14. Publication figures

The figures are descriptive summaries of the fixed analyses. They do not imply a clinically optimal intervention time.


In [ ]:
plot_configs = [
    "PRE",
    "CUM",
    "DYN",
    "CUM+DYN",
    "ALL",
]

fig, ax = plt.subplots(figsize=(9, 5.5))

for config in plot_configs:
    tmp = (
        ablation_results.loc[
            ablation_results["configuration"] == config
        ]
        .sort_values("landmark")
    )

    ax.plot(
        tmp["landmark"],
        tmp["roc_auc"],
        marker="o",
        linewidth=2,
        label=config,
    )

ax.axhline(
    0.5,
    linestyle="--",
    linewidth=1,
)

ax.set_xlabel("Within-session landmark (min)")
ax.set_ylabel("ROC-AUC")
ax.set_title(
    "Landmark Discrimination by Feature Representation"
)
ax.set_xticks(LANDMARKS)
ax.legend(frameon=False, ncol=3)

fig.tight_layout()

fig.savefig(
    FIGURE_DIR / "landmark_feature_representations.png",
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    FIGURE_DIR / "landmark_feature_representations.pdf",
    bbox_inches="tight",
)

plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.5))

tmp = (
    primary_uncertainty
    .sort_values("landmark")
)

x = tmp["landmark"].to_numpy()
y = tmp["roc_auc"].to_numpy()

lower = y - tmp["roc_ci_low"].to_numpy()
upper = tmp["roc_ci_high"].to_numpy() - y

ax.errorbar(
    x,
    y,
    yerr=[lower, upper],
    fmt="o-",
    linewidth=2,
    capsize=6,
)

ax.axhline(
    0.5,
    linestyle="--",
    linewidth=1,
)

ax.set_xlabel("Within-session landmark (min)")
ax.set_ylabel("ROC-AUC")
ax.set_title(
    "Primary CUM+DYN Discrimination with Athlete-Cluster Uncertainty"
)
ax.set_xticks(LANDMARKS)

fig.tight_layout()

fig.savefig(
    FIGURE_DIR / "primary_landmark_uncertainty.png",
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    FIGURE_DIR / "primary_landmark_uncertainty.pdf",
    bbox_inches="tight",
)

plt.show()


## 15. Export canonical results and provenance artifacts

In [ ]:
# Core tables
primary_results.to_csv(
    OUTPUT_DIR / "primary_logistic_results.csv",
    index=False,
)

primary_uncertainty.to_csv(
    OUTPUT_DIR / "primary_logistic_cluster_bootstrap.csv",
    index=False,
)

common_cohort_results.to_csv(
    OUTPUT_DIR / "common_cohort_results.csv",
    index=False,
)

alternative_fold_results.to_csv(
    OUTPUT_DIR / "alternative_negative_fold_allocations_raw.csv",
    index=False,
)

alternative_fold_summary.to_csv(
    OUTPUT_DIR / "alternative_negative_fold_allocations_summary.csv",
    index=False,
)

equal_athlete_results.to_csv(
    OUTPUT_DIR / "equal_athlete_weight_results.csv",
    index=False,
)

ablation_results.to_csv(
    OUTPUT_DIR / "feature_ablation_results.csv",
    index=False,
)

benchmark_results.to_csv(
    OUTPUT_DIR / "model_benchmark_results.csv",
    index=False,
)

paired_ablation_results.to_csv(
    OUTPUT_DIR / "paired_ablation_bootstrap.csv",
    index=False,
)

paired_model_results.to_csv(
    OUTPUT_DIR / "paired_model_bootstrap.csv",
    index=False,
)

fold_assignment.to_csv(
    OUTPUT_DIR / "outer_fold_athlete_assignments.csv",
    index=False,
)

landmark_availability.to_csv(
    OUTPUT_DIR / "landmark_availability.csv",
    index=False,
)

# OOF predictions
primary_oof_export = pd.concat(
    [
        frame.assign(landmark=landmark)
        for landmark, frame in primary_oof.items()
    ],
    ignore_index=True,
)

primary_oof_export.to_csv(
    OUTPUT_DIR / "primary_logistic_oof_predictions.csv",
    index=False,
)

benchmark_oof_export = pd.concat(
    [
        frame.assign(
            landmark=landmark,
            model=model_name,
        )
        for model_name, landmark_dict in benchmark_oof.items()
        for landmark, frame in landmark_dict.items()
    ],
    ignore_index=True,
)

benchmark_oof_export.to_csv(
    OUTPUT_DIR / "model_benchmark_oof_predictions.csv",
    index=False,
)

ablation_oof_export = pd.concat(
    [
        frame.assign(
            landmark=landmark,
            configuration=config_name,
        )
        for config_name, landmark_dict in ablation_oof.items()
        for landmark, frame in landmark_dict.items()
    ],
    ignore_index=True,
)

ablation_oof_export.to_csv(
    OUTPUT_DIR / "feature_ablation_oof_predictions.csv",
    index=False,
)

runtime_versions = pd.DataFrame(
    [
        {"package": "python", "version": platform.python_version()},
        {"package": "numpy", "version": np.__version__},
        {"package": "pandas", "version": pd.__version__},
        {"package": "scikit-learn", "version": sklearn.__version__},
        {
            "package": "xgboost",
            "version": (
                xgboost.__version__
                if XGBOOST_AVAILABLE
                else "not installed"
            ),
        },
    ]
)

runtime_versions.to_csv(
    OUTPUT_DIR / "runtime_versions.csv",
    index=False,
)

print("Canonical Notebook 04 artifacts saved to:", OUTPUT_DIR)


## Output contract and claims discipline

A successful run exports the primary Logistic Regression analysis, athlete-cluster uncertainty, common-cohort sensitivity, 100 alternative negative-athlete fold allocations, equal-athlete weighting, feature-family ablations, current-environment model benchmarks, paired bootstrap contrasts, fold assignments, runtime versions, and all canonical OOF predictions.

Interpretation must remain bounded by the data:

- only five athletes contribute positive sessions;
- the 22 positive athlete-sessions are not necessarily 22 independent confirmed injury events;
- exact within-session injury onset is unknown;
- Team B contains no positive session and is not an injury-positive external validation cohort;
- bootstrap intervals are conditional on the fitted cross-validation procedure;
- model and feature contrasts are descriptive sensitivity analyses rather than multiplicity-adjusted confirmatory discoveries;
- no result supports minute-specific prospective injury-risk estimation, injury-onset localization, or causal interpretation.
